In [1]:
%load_ext autoreload
%autoreload 2
%set_env TOKENIZERS_PARALLELISM=false
# for i, t in enumerate(SEQUENCE_VOCAB):
#     assert model.tokenizer.get_vocab()[t] == i

env: TOKENIZERS_PARALLELISM=false


In [ ]:
import os, json, pickle
import numpy as np
import torch
device = torch.device('cuda:0')
seq = "MKAKELREKSVEELNTELLNLLREQFNLRMQAASGQLQQSHLLKQVRRDVARVKTLLNEKAGA" 

from go_ml.esm3.viz_utils import get_pssm_display
import bokeh as bh
import bokeh.plotting
bokeh.io.output_notebook()

# from esm.sdk.api import (
#     ESMProtein,
#     GenerationConfig,
# )

# from esm.models.esmc import ESMC
# model = ESMC.from_pretrained("esmc_600m").to(device).eval() # or "cpu"


### ESM2

In [46]:
from transformers import AutoModelForMaskedLM
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, AutoModelForSeq2SeqLM, AutoConfig

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
config = AutoConfig.from_pretrained('facebook/esm2_t33_650M_UR50D')
model = AutoModelForMaskedLM.from_pretrained('facebook/esm2_t33_650M_UR50D').to(device).eval()

SEQUENCE_MASK_TOKEN = tokenizer.mask_token_id

from go_ml.masking import *
def get_logits(seq, batch_size=8, mask_func=mask_indiv):
    seq_ind = torch.LongTensor(tokenizer.encode(seq)).to(device)
    ln = len(seq)
    batch, batch_inds, mut_inds = mask_func(seq_ind, SEQUENCE_MASK_TOKEN)
    bert_eval_l = []
    with torch.no_grad():
        for si in range(0, batch.shape[0], batch_size):
            ei = min(batch.shape[0], si+batch_size)
            x = batch[si:ei, :]
            model_eval = model(x)
            bert_eval = model_eval.logits
            bert_eval_l.append(bert_eval.cpu())
    bert_eval = torch.cat(bert_eval_l)
    bert_eval = torch.softmax(bert_eval, dim=2)
    bert_mask = (batch == SEQUENCE_MASK_TOKEN).cpu()
    eval_avg, eval_support = mask_avg(bert_mask, bert_eval)
    # eval_avg = torch.softmax(eval_avg, dim=1)
    return eval_avg

logits = get_logits(seq, batch_size=16, mask_func=mask_indiv).float()[:, 4:24]
# pssm = logits.float()[:, 4:24]

In [47]:
# disp = get_pssm_display(seq, pssm)
# bh.plotting.show(disp)
from go_ml.esm3.viz_utils import pssm_to_dataframe, esm_alphabet
pssm_df = pssm_to_dataframe(logits, esm_alphabet)
num_colors = 256  # You can adjust this number
palette = bh.palettes.viridis(256)
TOOLS = "hover,save,pan,box_zoom,reset,wheel_zoom"
p = bh.plotting.figure(title="CONSERVATION",
        x_range=[str(x) for x in range(1,len(seq)+1)],
        y_range=list(esm_alphabet)[::-1],
        width=900, height=400,
        tools=TOOLS, toolbar_location='below',
        tooltips=[('Position', '@Position'), ('Amino Acid', '@{Amino Acid}'), ('Probability', '@Probability')])

r = p.rect(x="Position", y="Amino Acid", width=1, height=1, source=pssm_df,
        fill_color=bh.transform.linear_cmap('Probability', palette, low=0, high=0.5),
        line_color=None)
p.xaxis.visible = False  # Hide the x-axis
bh.plotting.show(p)

### ESM3

In [24]:
from esm.utils.constants.esm3 import SEQUENCE_VOCAB
from esm.models.esm3 import ESM3
from esm.sdk.api import (
    ESMProtein,
    GenerationConfig,
)

model =  ESM3.from_pretrained("esm3_sm_open_v1", device=torch.device(device)).eval()

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

In [26]:
protein_prompt = ESMProtein(sequence=seq)
protein_tensor = model.encode(protein_prompt)
x, ln = protein_tensor.sequence, len(seq)
x = x.reshape(1, -1)

def model_eval(x):
  with(torch.no_grad(), torch.autocast(enabled=True, device_type=device.type, dtype=torch.bfloat16)):
    logits = model.forward(sequence_tokens=x)
    return logits.sequence_logits[:, 1:(ln+1), 4:24].detach().cpu().float().numpy()

import tqdm.notebook
TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'
def get_logits(seq, batch_size=1):
  protein_prompt = ESMProtein(sequence=seq)
  protein_tensor = model.encode(protein_prompt)
  x, ln = protein_tensor.sequence, len(seq)
  with torch.no_grad():
    logits = np.zeros((ln, 20), dtype=np.float32)
    with tqdm.notebook.tqdm(total=ln, bar_format=TQDM_BAR_FORMAT) as pbar:
      for n in range(0, ln, batch_size):
        m = min(n + batch_size, ln)
        x_h = torch.clone(x).unsqueeze(0).repeat(m - n, 1)
        for i in range(m - n):
          x_h[i, n + i + 1] = SEQUENCE_VOCAB.index("<mask>")
        fx_h = model_eval(x_h.to(device))
        for i in range(m - n):
          logits[n + i] = fx_h[i, n + i]
        pbar.update(m - n)
  return logits

esm3_logits = get_logits(seq, batch_size=8)

  0%|          | 0/63 [elapsed: 00:00 remaining: ?]

In [29]:
disp = get_pssm_display(seq, esm3_logits)
bh.plotting.show(disp)

### ESMC

In [30]:
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
from esm.utils.constants.esm3 import (
    SEQUENCE_MASK_TOKEN,
)
import torch
device = torch.device('cuda:0')
model = ESMC.from_pretrained("esmc_600m").to(device) # or "cpu"

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [31]:
from go_ml.masking import *
def get_logits(seq, batch_size=8, mask_func=mask_indiv):
    seq_ind = model.encode(ESMProtein(sequence=seq)).sequence
    batch, batch_inds, mut_inds = mask_func(seq_ind, SEQUENCE_MASK_TOKEN)
    bert_eval_l = []
    with torch.no_grad():
        for si in range(0, batch.shape[0], batch_size):
            ei = min(batch.shape[0], si+batch_size)
            x = batch[si:ei, :]
            model_eval = model(x)
            bert_eval = model_eval.sequence_logits
            bert_eval_l.append(bert_eval.cpu())
    bert_eval = torch.cat(bert_eval_l)
    bert_eval = torch.softmax(bert_eval, dim=2)
    bert_mask = (batch == SEQUENCE_MASK_TOKEN).cpu()
    eval_avg, eval_support = mask_avg(bert_mask, bert_eval)
    return eval_avg

esmc_logits = get_logits(seq, batch_size=16, mask_func=mask_indiv)

In [32]:
disp = get_pssm_display(seq, esm3_logits)
bh.plotting.show(disp)